In [38]:

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pandas_datareader as web
import datetime as dt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, InputLayer

In [39]:
#------------------------------------------------------------------------------
# Load Data
#------------------------------------------------------------------------------
import os
import yfinance as yf
import pandas as pd

COMPANY = 'CBA.AX'
TRAIN_START = '2020-01-01'
TRAIN_END = '2023-08-01'

DATA_DIR = "data"
CSV_FILE = os.path.join(DATA_DIR, f"{COMPANY}_stock_data.csv")

# Ensure data directory exists
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

# Check if file exists
if os.path.exists(CSV_FILE):
    print(f"Loading saved data from {CSV_FILE}...")
    data = pd.read_csv(CSV_FILE, index_col="Date", parse_dates=True)
else:
    print("Downloading data from Yahoo Finance...")
    data = yf.download(COMPANY, start=TRAIN_START, end=TRAIN_END)
    data.to_csv(CSV_FILE)
    print(f"Data saved to {CSV_FILE}")

print(data.head())


/var/folders/n3/f5ytszns27bb7c95jm3qrfdh0000gn/T/ipykernel_47057/73977203.py:25: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(COMPANY, start=TRAIN_START, end=TRAIN_END)
[*********************100%***********************]  1 of 1 completed

Data saved to data/CBA.AX_stock_data.csv
Price           Close       High        Low       Open   Volume
Ticker         CBA.AX     CBA.AX     CBA.AX     CBA.AX   CBA.AX
Date                                                           
2020-01-02  64.933327  65.209711  64.535014  64.860170  1416232
2020-01-03  65.282867  65.998211  65.234096  65.819375  1622784
2020-01-06  64.843903  64.933321  64.404950  64.811393  2129260
2020-01-07  66.006332  66.006332  65.177192  65.697438  2417468
2020-01-08  65.762451  66.046960  65.055238  66.022574  1719114


In [43]:
#------------------------------------------------------------------------------
# Prepare Data
#------------------------------------------------------------------------------
PREPARED_DATA_DIR = "prepared_data"
PREPARED_CSV = os.path.join(PREPARED_DATA_DIR, f"{COMPANY}_prepared.csv")
PREDICTION_DAYS = 60 # You can change the number of days to predict ahead

# Ensure directory exists
if not os.path.exists(PREPARED_DATA_DIR):
    os.makedirs(PREPARED_DATA_DIR)

# Check if prepared data exists
if os.path.exists(PREPARED_CSV):
    print(f"Loading prepared data from {PREPARED_CSV}...")
    prepared_data = pd.read_csv(PREPARED_CSV, index_col="Date", parse_dates=True)
else:
    print("Preparing data...")
    prepared_data = data.copy()
    
    # Example: Use mid-point of Open & Close as price
    prepared_data['MidPrice'] = (prepared_data['Open'] + prepared_data['Close']) / 2
    
    # You can shift the price column to create target for prediction
    prepared_data[f"FuturePrice_{PREDICTION_DAYS}d"] = prepared_data['MidPrice'].shift(-PREDICTION_DAYS)
    
    # Drop rows with NaN values (will occur at the end due to shift)
    prepared_data.dropna(inplace=True)
    
    # Save the prepared data
    prepared_data.to_csv(PREPARED_CSV)
    print(f"Prepared data saved to {PREPARED_CSV}")

print(prepared_data.head())


Loading prepared data from prepared_data/CBA.AX_prepared.csv...


ValueError: 'Date' is not in list

In [41]:
PRICE_VALUE = "Close"
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data[PRICE_VALUE].values.reshape(-1, 1))
scaled_data = scaled_data[:, 0]  # Flatten to 1D


In [42]:
PREDICTION_DAYS = 60  # Number of past days the model looks at
x_train, y_train = [], []

for i in range(PREDICTION_DAYS, len(scaled_data)):
    x_train.append(scaled_data[i-PREDICTION_DAYS:i])
    y_train.append(scaled_data[i])

x_train, y_train = np.array(x_train), np.array(y_train)
x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))


In [ ]:
#-------------------- Build the LSTM Model --------------------#
model = Sequential()

# Input layer: shape matches number of past days used for prediction
model.add(InputLayer(input_shape=(x_train.shape[1], 1)))

# First LSTM layer with Dropout for regularization
model.add(LSTM(units=50, return_sequences=True))
model.add(Dropout(0.2))

# Second LSTM layer with Dropout
model.add(LSTM(units=50))
model.add(Dropout(0.2))

# Output layer: predicts a single value (next price)
model.add(Dense(1))

# Compile the model with optimizer and loss function
model.compile(optimizer='adam', loss='mean_squared_error')

#-------------------- Train the Model --------------------#
# We train the model on x_train, y_train
# validation_split=0.1 uses 10% of the training data for validation
# verbose=1 prints training progress
history = model.fit(
    x_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

#-------------------- Save the Model --------------------#
# After training, save the model to reuse later without retraining
model.save("results/lstm_stock_model.h5")


/Users/lasithcharuka/hand/stockpred-env/lib/python3.13/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Epoch 1/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - loss: 0.0840 - val_loss: 0.0282
Epoch 2/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - loss: 0.0116 - val_loss: 0.0069
Epoch 3/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - loss: 0.0081 - val_loss: 9.5523e-04
Epoch 4/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 0.0062 - val_loss: 9.4752e-04
Epoch 5/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - loss: 0.0057 - val_loss: 0.0028
Epoch 6/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - loss: 0.0055 - val_loss: 0.0026
Epoch 7/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - loss: 0.0054 - val_loss: 9.5358e-04
Epoch 8/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - loss: 0.0055 - val_loss: 9.6694e-04
Epoch 9/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - loss: 0.0057 - val_loss: 0.0022
Epoch 10/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - loss: 0.0052 - val_loss: 0.0025
Epoch 11/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - loss: 0.0057 - val_loss: 0.0010
Epoch 12/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/

In [ ]:
#------------------------------------------------------------------------------  
# Test the model accuracy on existing data  
#------------------------------------------------------------------------------  

# Define the test period
TEST_START = '2023-08-02'
TEST_END = '2024-07-02'

# Download the test data using yfinance
test_data = yf.download(COMPANY, TEST_START, TEST_END)

# Get actual closing prices from the test set
actual_prices = test_data[PRICE_VALUE].values

# Combine the training and test data for consistent scaling
total_dataset = pd.concat((data[PRICE_VALUE], test_data[PRICE_VALUE]), axis=0)

# Prepare inputs for the model: we need the last PREDICTION_DAYS from training
# + all the test data for making predictions
model_inputs = total_dataset[len(total_dataset) - len(test_data) - PREDICTION_DAYS:].values

# Reshape to a 2D array as required by the scaler (n_samples, 1)
model_inputs = model_inputs.reshape(-1, 1)

# Scale the input values using the same scaler as for training
# This ensures the model input is in the same 0-1 range as during training
model_inputs = scaler.transform(model_inputs)

# ------------------ Notes / Explanations ------------------  
# 1. We need the last PREDICTION_DAYS of the training data to predict the first day 
#    of the test period because the model uses sequences of length PREDICTION_DAYS.  
# 2. Reshape(-1,1) converts a 1D array to 2D so that MinMaxScaler can process it.  
# 3. ISSUE #2: If test prices exceed the training min/max, scaled values can go below 0 or above 1.  
#    A better approach: fit the scaler on the combined training+test set or use rolling normalization.  


/var/folders/n3/f5ytszns27bb7c95jm3qrfdh0000gn/T/ipykernel_47057/3616154971.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  test_data = yf.download(COMPANY, TEST_START, TEST_END)
[*********************100%***********************]  1 of 1 completed


In [ ]:
# Prepare test sequences for the LSTM model
x_test = []
for x in range(PREDICTION_DAYS, len(model_inputs)):
    # Take the previous PREDICTION_DAYS values as one input sequence
    x_test.append(model_inputs[x - PREDICTION_DAYS:x, 0])

# Convert the list into a NumPy array
x_test = np.array(x_test)

# Reshape into 3D shape: (samples, time steps, features)
# This is the input format LSTM expects
x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], 1))

# Predict using the trained model
predicted_prices = model.predict(x_test)

# Inverse transform the scaled predictions to get actual stock prices
predicted_prices = scaler.inverse_transform(predicted_prices)


8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step
